In [2]:
import pandas as pd
from sqlalchemy import create_engine, inspect
import seaborn as sns
import matplotlib.pyplot as plt
def get_database(database_address):
    database_address = database_address
    schema_name = ""

    engine = create_engine(database_address)

    inspector = inspect(engine)
    tables = inspector.get_table_names(schema=schema_name)

**Part 1: Getting Data from SQL**

In [28]:
"""
Retrieving patient data
"""
query = """
SELECT "ANESTHESIA_RECORD"."CASE_ID", "ANESTHESIA_RECORD"."AN_EPISODE_ID", "ANESTHESIA_RECORD"."PAT_ENC_CSN_ID", "ANESTHESIA_RECORD"."ASA_SCORE_C", "ANESTHESIA_RECORD"."AN_PROC_NAME", "ANESTHESIA_RECORD"."IN_PACU_DTTM", "ANESTHESIA_RECORD"."FIRST_PACU_PAIN_SCORE", "ANESTHESIA_RECORD"."HIGHEST_PACU_PAIN_SCORE" FROM "ANESTHESIA_RECORD" WHERE "IN_PACU_DTTM" > '2021-12-31';
"""
patient_data = pd.read_sql(query, engine);
patient_data.dropna(inplace = True);

In [29]:
"""
Filtering for T&A patients
"""
pattern = r'([^;]*(\bTONSILLECTOMY\b|\bADENOIDECTOMY\b|\bT&A\b)[^;]*;?)+'
t_and_a_patient_data = patient_data[patient_data['AN_PROC_NAME'].str.fullmatch(pattern)]

In [30]:
"""
Retrieve location data
"""

query = """
SELECT "LOCATION", "CASE_ID" from "OR_SURGERY"
"""

location_data = pd.read_sql(query, engine)

In [31]:
location_data_merged = pd.merge(t_and_a_patient_data, location_data, on="CASE_ID", how="left")

In [74]:
"""
Dividing by location
"""
t_and_a_patient_data_MAIN = location_data_merged[location_data_merged["LOCATION"] == 'PERIOP COMPLEX']
t_and_a_patient_data_ASC = location_data_merged[location_data_merged["LOCATION"].isin(['VOORHEES DAY SURGERY', 'BUCKS DAY SURGERY', 'BRANDYWINE VALLEY DAY SURGERY'])]
t_and_a_patient_data_KOP = location_data_merged[location_data_merged["LOCATION"].isin(['KING OF PRUSSIA HOSPITAL', "KING OF PRUSSIA DAY SURGERY"])]
t_and_a_patient_data_VOORHEES = location_data_merged[location_data_merged["LOCATION"].isin(['VOORHEES DAY SURGERY'])]
t_and_a_patient_data_BUCKS = location_data_merged[location_data_merged["LOCATION"].isin(['BUCKS DAY SURGERY'])]
t_and_a_patient_data_BRANDYWINE = location_data_merged[location_data_merged["LOCATION"].isin(['BRANDYWINE VALLEY DAY SURGERY'])]

In [76]:
"""
Retrieving patient demographic data
"""
patient_ids = tuple(t_and_a_patient_data['PAT_ENC_CSN_ID'].tolist())

query = f"""
SELECT "PAT_ENC_CSN_ID", "BIRTH_DATE", "GENDER", "RACE" FROM "DEMOGRAPHIC" WHERE "DEMOGRAPHIC"."PAT_ENC_CSN_ID" IN {patient_ids}
"""

demographic_data = pd.read_sql(query, engine)
demographic_data.dropna(inplace=True)

demographic_data = demographic_data.groupby('PAT_ENC_CSN_ID').agg({
    "BIRTH_DATE":'first',
    "GENDER":'first',
    "RACE": lambda x: ','.join(sorted(set(x)))
}
)

In [77]:
"""
Retrieving patient medication data
"""

medications = ['%MORPHINE%', '%FENTANYL%', '%ACETAMINOPHEN%', '%DEXMEDETOMIDINE%', '%DEXAMETHASONE%', "%KETAMINE%", "%CELECOXIB%"]
query = """
SELECT DISTINCT "AN_EPISODE_ID", "PAT_MRN_ID", "PRE_INTRA_POST_OP", "AN_START_DATETIME", "AN_STOP_DATETIME", "MED_NAME", "DOSE", "SITE", "UNIT", "ROUTE", "RESULT", "STRENGTH", "INFSN_RATE"
FROM "PERI_OP_MED" WHERE "MED_NAME" ILIKE ANY (%s) AND "PRE_INTRA_POST_OP" = 'IntraOp';
"""

peri_op_med = pd.read_sql(query, engine, params=(medications,))

peri_op_med_condensed = peri_op_med.groupby("AN_EPISODE_ID").apply(
    lambda group: pd.Series({
        'medications': ','.join(med for med in sorted(set(group['MED_NAME']))),
        'route': ','.join(sorted(set(rt for rt in group['ROUTE'] if pd.notnull(rt))))
    }), include_groups = False
).reset_index().copy()

In [78]:
"""
Exclusion criteria: patients not given dexamethasone, patients given ketamine or celecoxib
"""
#exclude patients who were not given dexamethasone
peri_op_med_condensed = peri_op_med_condensed[peri_op_med_condensed["medications"].str.contains("DEXAMETHASONE")]
print(len(peri_op_med_condensed))
peri_op_med_condensed = peri_op_med_condensed[peri_op_med_condensed["medications"].str.contains("CELECOXIB") == False]
print(len(peri_op_med_condensed))
peri_op_med_condensed = peri_op_med_condensed[peri_op_med_condensed["medications"].str.contains("KETAMINE") == False]

130330
130087


In [79]:
"""
Filter for only IV medications
"""

peri_op_med_condensed = peri_op_med_condensed[peri_op_med_condensed['route'] == "Intravenous"]

In [80]:
"""
Combining all patient data into one DataFrame
"""
t_and_a_patient_data_peri_op_MAIN = pd.merge(t_and_a_patient_data_MAIN, peri_op_med_condensed, on="AN_EPISODE_ID", how="left")
t_and_a_patient_data_peri_op_MAIN = pd.merge(t_and_a_patient_data_peri_op_MAIN, demographic_data, on="PAT_ENC_CSN_ID", how="left")
t_and_a_patient_data_peri_op_MAIN = t_and_a_patient_data_peri_op_MAIN.dropna(inplace = False)

t_and_a_patient_data_peri_op_ASC = pd.merge(t_and_a_patient_data_ASC, peri_op_med_condensed, on="AN_EPISODE_ID", how="left")
t_and_a_patient_data_peri_op_ASC = pd.merge(t_and_a_patient_data_peri_op_ASC, demographic_data, on="PAT_ENC_CSN_ID", how="left")
t_and_a_patient_data_peri_op_ASC = t_and_a_patient_data_peri_op_ASC.dropna(inplace = False)

t_and_a_patient_data_peri_op_KOP = pd.merge(t_and_a_patient_data_KOP, peri_op_med_condensed, on="AN_EPISODE_ID", how="left")
t_and_a_patient_data_peri_op_KOP = pd.merge(t_and_a_patient_data_peri_op_KOP, demographic_data, on="PAT_ENC_CSN_ID", how="left")
t_and_a_patient_data_peri_op_KOP = t_and_a_patient_data_peri_op_KOP.dropna(inplace = False)

t_and_a_patient_data_peri_op_VOORHEES = pd.merge(t_and_a_patient_data_VOORHEES, peri_op_med_condensed, on="AN_EPISODE_ID", how="left")
t_and_a_patient_data_peri_op_VOORHEES = pd.merge(t_and_a_patient_data_peri_op_VOORHEES, demographic_data, on="PAT_ENC_CSN_ID", how="left")
t_and_a_patient_data_peri_op_VOORHEES = t_and_a_patient_data_peri_op_VOORHEES.dropna(inplace = False)

t_and_a_patient_data_peri_op_BUCKS = pd.merge(t_and_a_patient_data_BUCKS, peri_op_med_condensed, on="AN_EPISODE_ID", how="left")
t_and_a_patient_data_peri_op_BUCKS = pd.merge(t_and_a_patient_data_peri_op_BUCKS, demographic_data, on="PAT_ENC_CSN_ID", how="left")
t_and_a_patient_data_peri_op_BUCKS = t_and_a_patient_data_peri_op_BUCKS.dropna(inplace = False)

t_and_a_patient_data_peri_op_BRANDYWINE = pd.merge(t_and_a_patient_data_BRANDYWINE, peri_op_med_condensed, on="AN_EPISODE_ID", how="left")
t_and_a_patient_data_peri_op_BRANDYWINE = pd.merge(t_and_a_patient_data_peri_op_BRANDYWINE, demographic_data, on="PAT_ENC_CSN_ID", how="left")
t_and_a_patient_data_peri_op_BRANDYWINE = t_and_a_patient_data_peri_op_BRANDYWINE.dropna(inplace = False)


**Step 3: Cleaning Race Column and Calculating Age**

- if we have 2 different races, in which one race is "Unknown", the other race was chosen
- if we have 2+ different races, race is classified as "Unknown"


In [82]:
def clean_races(row):
    unknown_category = ["Other", "Refused", "Choose not to disclose", "Unknown", "Asked but unknown"]

    if pd.isna(row["RACE"]):
        return "Unknown"

    new_race = row["RACE"].split(",")
    if len(new_race) > 1:
        if len(new_race) == 2:
            if new_race[0] in unknown_category:
                new_race = [new_race[1]]
            elif new_race[1] in unknown_category:
                new_race = [new_race[0]]
        elif len(new_race) > 2:
            new_race = ["Multi-race"]

    new_race = new_race[0]
    if new_race != "Multi-race" and new_race in unknown_category:
        new_race = "Unknown"

    return new_race

In [83]:
t_and_a_patient_data_peri_op_ASC["RACE_cleaned"] = t_and_a_patient_data_peri_op_ASC.apply(clean_races, axis = 1)

In [84]:
t_and_a_patient_data_peri_op_MAIN["RACE_cleaned"] = t_and_a_patient_data_peri_op_MAIN.apply(clean_races, axis = 1)

In [85]:
t_and_a_patient_data_peri_op_KOP["RACE_cleaned"] = t_and_a_patient_data_peri_op_KOP.apply(clean_races, axis = 1)

In [86]:
t_and_a_patient_data_peri_op_VOORHEES["RACE_cleaned"] = t_and_a_patient_data_peri_op_VOORHEES.apply(clean_races, axis = 1)
t_and_a_patient_data_peri_op_BRANDYWINE["RACE_cleaned"] = t_and_a_patient_data_peri_op_BRANDYWINE.apply(clean_races, axis = 1)
t_and_a_patient_data_peri_op_BUCKS["RACE_cleaned"] = t_and_a_patient_data_peri_op_BUCKS.apply(clean_races, axis = 1)

In [87]:
"""
Calculating Age
"""

today = pd.Timestamp.today()

t_and_a_patient_data_peri_op_ASC["BIRTH_DATE_datetime"] = pd.to_datetime(t_and_a_patient_data_peri_op_ASC["BIRTH_DATE"], errors='coerce')
t_and_a_patient_data_peri_op_ASC['age'] = t_and_a_patient_data_peri_op_ASC["BIRTH_DATE_datetime"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))

t_and_a_patient_data_peri_op_MAIN["BIRTH_DATE_datetime"] = pd.to_datetime(t_and_a_patient_data_peri_op_MAIN["BIRTH_DATE"], errors='coerce')
t_and_a_patient_data_peri_op_MAIN['age'] = t_and_a_patient_data_peri_op_MAIN["BIRTH_DATE_datetime"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))

t_and_a_patient_data_peri_op_KOP["BIRTH_DATE_datetime"] = pd.to_datetime(t_and_a_patient_data_peri_op_KOP["BIRTH_DATE"], errors='coerce')
t_and_a_patient_data_peri_op_KOP['age'] = t_and_a_patient_data_peri_op_KOP["BIRTH_DATE_datetime"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))

t_and_a_patient_data_peri_op_VOORHEES["BIRTH_DATE_datetime"] = pd.to_datetime(t_and_a_patient_data_peri_op_VOORHEES["BIRTH_DATE"], errors='coerce')
t_and_a_patient_data_peri_op_VOORHEES['age'] = t_and_a_patient_data_peri_op_VOORHEES["BIRTH_DATE_datetime"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))

t_and_a_patient_data_peri_op_BRANDYWINE["BIRTH_DATE_datetime"] = pd.to_datetime(t_and_a_patient_data_peri_op_BRANDYWINE["BIRTH_DATE"], errors='coerce')
t_and_a_patient_data_peri_op_BRANDYWINE['age'] = t_and_a_patient_data_peri_op_BRANDYWINE["BIRTH_DATE_datetime"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))

t_and_a_patient_data_peri_op_BUCKS["BIRTH_DATE_datetime"] = pd.to_datetime(t_and_a_patient_data_peri_op_BUCKS["BIRTH_DATE"], errors='coerce')
t_and_a_patient_data_peri_op_BUCKS['age'] = t_and_a_patient_data_peri_op_BUCKS["BIRTH_DATE_datetime"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))

In [88]:
"""
Calculating age groups to look only at patients 1 - 18
"""
bins=[0,2,5,11,18]
labels = [
    '1 yr. - 2 yr.',
    '3 yr. - 5 yr.',
    '6 yr. - 11 yr.',
    '12 yr. - 18 yr.',
]

t_and_a_patient_data_peri_op_ASC['age_group'] = pd.cut(t_and_a_patient_data_peri_op_ASC['age'], bins=bins, labels=labels)
t_and_a_patient_data_peri_op_ASC = t_and_a_patient_data_peri_op_ASC.dropna(inplace = False)

t_and_a_patient_data_peri_op_MAIN['age_group'] = pd.cut(t_and_a_patient_data_peri_op_MAIN['age'], bins=bins, labels=labels)
t_and_a_patient_data_peri_op_MAIN = t_and_a_patient_data_peri_op_MAIN.dropna(inplace = False)

t_and_a_patient_data_peri_op_KOP['age_group'] = pd.cut(t_and_a_patient_data_peri_op_KOP['age'], bins=bins, labels=labels)
t_and_a_patient_data_peri_op_KOP = t_and_a_patient_data_peri_op_KOP.dropna(inplace = False)

t_and_a_patient_data_peri_op_VOORHEES['age_group'] = pd.cut(t_and_a_patient_data_peri_op_VOORHEES['age'], bins=bins, labels=labels)
t_and_a_patient_data_peri_op_VOORHEES = t_and_a_patient_data_peri_op_VOORHEES.dropna(inplace = False)

t_and_a_patient_data_peri_op_BRANDYWINE['age_group'] = pd.cut(t_and_a_patient_data_peri_op_BRANDYWINE['age'], bins=bins, labels=labels)
t_and_a_patient_data_peri_op_BRANDYWINE = t_and_a_patient_data_peri_op_BRANDYWINE.dropna(inplace = False)

t_and_a_patient_data_peri_op_BUCKS['age_group'] = pd.cut(t_and_a_patient_data_peri_op_BUCKS['age'], bins=bins, labels=labels)
t_and_a_patient_data_peri_op_BUCKS = t_and_a_patient_data_peri_op_BUCKS.dropna(inplace = False)

**Step 4: Filtering for regimes/medication groups that we're interested in**

In [89]:
regimens = [[['acetaminophen'], ["fentanyl", "dexmedetomidine", 'morphine']],
            [['dexmedtomidine'], ["fentanyl", "acetaminophen", "morphine"]],
            [['morphine'], ["fentanyl", "dexmedetomidine", 'acetaminophen']],
            [['fentanyl'], ["morphine", "dexmedetomidine", 'acetaminophen']],
            [['dexmedetomidine', 'acetaminophen'], ["fentanyl", "morphine"]],
            [['morphine', 'acetaminophen'], ["fentanyl", "dexmedetomidine"]],
            [['fentanyl', 'acetaminophen'], ["morphine", 'dexmedetomidine']],
            [['morphine', 'dexmedetomidine'], ["fentanyl", 'acetaminophen']],
            [['fentanyl', "dexmedetomidine"], ["morphine", 'acetaminophen']],
            [['morphine', 'dexmedetomidine', 'acetaminophen'], ["fentanyl"]],
            [['fentanyl', "dexmedetomidine", 'acetaminophen'], ["morphine"]]]


In [90]:
"""
Create different medication combinations
"""
def create_combinations(df, meds_included, meds_excluded, med_combo_name):
    pts_w_med_combo = df[df['medications'].apply(lambda meds: all(kw in meds.lower() for kw in meds_included) and not any(kw in meds.lower() for kw in meds_excluded))]['AN_EPISODE_ID']

    df.loc[df['AN_EPISODE_ID'].isin(pts_w_med_combo), 'med_combos'] = med_combo_name

    return df

In [94]:
"""
Creating medication group label
"""
for index, combo in enumerate(regimens):
    abbreviation = ''.join(s[:3].capitalize() for i,s in enumerate(combo[0]))
    create_combinations(t_and_a_patient_data_peri_op_ASC, combo[0], combo[1], abbreviation)
    create_combinations(t_and_a_patient_data_peri_op_MAIN, combo[0], combo[1], abbreviation)
    create_combinations(t_and_a_patient_data_peri_op_KOP, combo[0], combo[1], abbreviation)
    create_combinations(t_and_a_patient_data_peri_op_BUCKS, combo[0], combo[1], abbreviation)
    create_combinations(t_and_a_patient_data_peri_op_BRANDYWINE, combo[0], combo[1], abbreviation)
    create_combinations(t_and_a_patient_data_peri_op_VOORHEES, combo[0], combo[1], abbreviation)

In [95]:
t_and_a_patient_data_peri_op_ASC.dropna(inplace=True, axis = 0)
t_and_a_patient_data_peri_op_MAIN.dropna(inplace=True, axis = 0)
t_and_a_patient_data_peri_op_KOP.dropna(inplace=True, axis = 0)
t_and_a_patient_data_peri_op_VOORHEES.dropna(inplace=True, axis = 0)
t_and_a_patient_data_peri_op_BRANDYWINE.dropna(inplace=True, axis = 0)
t_and_a_patient_data_peri_op_BUCKS.dropna(inplace=True, axis = 0)

In [96]:
"""
Exporting dataframes
"""
t_and_a_patient_data_peri_op_ASC.to_pickle("t&a_df_ASC.pk1")
t_and_a_patient_data_peri_op_MAIN.to_pickle("t&a_df_MAIN.pk1")
t_and_a_patient_data_peri_op_KOP.to_pickle("t&a_df_KOP.pk1")
t_and_a_patient_data_peri_op_VOORHEES.to_pickle("t&a_df_VOORHEES.pk1")
t_and_a_patient_data_peri_op_BRANDYWINE.to_pickle("t&a_df_BRANDYWINE.pk1")
t_and_a_patient_data_peri_op_BUCKS.to_pickle("t&a_df_BUCKS.pk1")

In [ ]:
"""
Using shapiro to check if pain score data is normally distributed
"""

from scipy.stats import shapiro

for regime, group in t_and_a_patient_data_peri_op.groupby("regime"):
    stat, p = shapiro(group["HIGHEST_PACU_PAIN_SCORE"])

    print(f"Regime: {regime}, W-stat={stat:.3f}, p={p:.3f}")

    if p> 0.05:
        print("Data is normal")
    else:
        print("Data is not normal")

"""
Pain score data is not normally distributed
"""

**Step 6: Graphing and IQR Calculation**

In [177]:
"""
Method to graph bar graph of statistics (mean, SD, median, IQR) for each medication combination
"""
def regimen_graph(regimen_df):
    patient_counts = regimen_df["med_combos"].value_counts()
    total_patients = sum(regimen_df["med_combos"].value_counts())
    median_vals = regimen_df.groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].median()
    mean_vals = regimen_df.groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].mean()
    std_vals = regimen_df.groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].std()

    first_quartile_1 = (regimen_df.groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].quantile(0.25))
    third_quartile_1 = (regimen_df.groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].quantile(0.75))
    iqr = third_quartile_1 - first_quartile_1

    fig, ax1 = plt.subplots(figsize=(14,6))
    sns.boxplot(data=regimen_df, x="med_combos", y="HIGHEST_PACU_PAIN_SCORE", hue="regime", order=["FenAce", "Mor", "MorAce", "MorDex", "MorDexAce", "Ace", "Fen", "DexAce", "FenDex", "FenDexAce"])
    ax1.set_xlabel("Medication Combinations")
    plt.xticks(rotation=40)
    ax1.set_ylabel("Average Pain Score")
    plt.title("Comparisons of Highest PACU pain scores among different analgesic combinations vs. Fentanyl + Acetaminophen", pad=10)
    plt.legend(title="Medication Combinations", bbox_to_anchor=(1,1), loc='upper left')

    for tick, label in enumerate(ax1.get_xticklabels()):
        med_group = label.get_text()
        count = patient_counts[med_group]
        median = median_vals[med_group]
        std = std_vals[med_group]
        mean = mean_vals[med_group]
        percentage = round(count / total_patients * 100, 1)

        y_position = regimen_df[regimen_df["med_combos"] == med_group]["HIGHEST_PACU_PAIN_SCORE"].max()
        median_position = regimen_df[regimen_df["med_combos"] == med_group]["HIGHEST_PACU_PAIN_SCORE"].mean()
        ax1.text(tick, y_position + 0.2, f"n={count}\n{percentage}%",
                 horizontalalignment="center",size="small", color="black")

        if med_group in ["FenAce", "Mor", "MorAce", "MorDexAce", "FenDex", "FenDexAce"]:
            ax1.text(tick, median_position + 1.0, f"median: {median}",
                     horizontalalignment="center", size="small", color="black")
            ax1.text(tick, median_position + 0.6, f"IQR: {iqr[med_group]}",
                     horizontalalignment="center", size="small", color="black")
            ax1.text(tick, median_position + 1.6, f"mean: {mean:.1f}",
                     horizontalalignment="center", size="small", color="black")
            ax1.text(tick, median_position + 1.3, f"STD: {std:.1f}",
                     horizontalalignment="center", size="small", color="black")
        else:
            ax1.text(tick, median_position - 0.9, f"median: {median}",
                     horizontalalignment="center", size="small", color="black")
            ax1.text(tick, median_position - 1.2, f"IQR: {iqr[med_group]}",
                     horizontalalignment="center", size="small", color="black")
            ax1.text(tick, median_position - 0.3, f"mean: {mean:.1f}",
                     horizontalalignment="center", size="small", color="black")
            ax1.text(tick, median_position - 0.6, f"STD: {std:.1f}",
                     horizontalalignment="center", size="small", color="black")


In [37]:
regimen_graph(regimen_df_all['All Demographics'],'All Demographics')

In [182]:
"""
Method to calculate IQR
"""
def iqr(x):
    if x.count() < 2:
        return None

    print("75%:", x.quantile(0.75))
    print("25%:", x.quantile(0.25))
    print("50%: ", x.quantile(0.5))
    return x.quantile(0.75) - x.quantile(0.25)

**Step 7: Investigating different demographics**

- ASA Value
- Gender
- Race
- Age group

In [216]:
def create_histogram(df, x_value):
    ax = sns.countplot(df, x=x_value)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    plt.xlabel(x_value)
    plt.ylabel("Number of Patients")
    plt.title("Histogram of " + x_value)

    for container in ax.containers:
        ax.bar_label(container)

In [217]:
def create_tabular_data_asa_value(df, asa_val):
    data = df[df["ASA_SCORE_C"] == asa_val]
    data = data["med_combos"].value_counts().reset_index()

    data = data.assign(name_length=data["med_combos"].str.len())
    data = data.sort_values(by="name_length").drop(columns="name_length")
    return data

In [218]:
def create_tabular_data_gender(df, gender):
    data = df[df["GENDER"] == gender]
    data = data["med_combos"].value_counts().reset_index()

    data = data.assign(name_length=data["med_combos"].str.len())
    data = data.sort_values(by="name_length").drop(columns="name_length")
    return data

In [108]:
def create_tabular_data_race(df, race):
    data = df[df["RACE_cleaned"] == race]
    data = data["med_combos"].value_counts().reset_index()

    data = data.assign(name_length=data["med_combos"].str.len())
    data = data.sort_values(by="name_length").drop(columns="name_length")
    return data

In [109]:
def create_tabular_data_age(df, age_group):
    data = df[df["age_group"] == age_group]
    data = data["med_combos"].value_counts().reset_index()

    data = data.assign(name_length=data["med_combos"].str.len())
    data = data.sort_values(by="name_length").drop(columns="name_length")
    return data

**EXAMINING ASA STATUS**

In [36]:
create_histogram(t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op]["ASA_SCORE_C"].isin([2.0,3.0,4.0])], "ASA_SCORE_C")

In [27]:
for asa_score in [2.0, 3.0, 4.0]:
    print("Number of Patients with ASA Score: "+ str(asa_score))
    print(create_tabular_data_asa_value(t_and_a_patient_data_peri_op, asa_score))

In [29]:
perform_posthoc(t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op["ASA_SCORE_C"] == 2.0])

In [30]:
t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op["ASA_SCORE_C"] == 2.0].groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].median()

**EXAMINING GENDER**

In [31]:
create_histogram(t_and_a_patient_data_peri_op, "GENDER")

In [32]:
for gender in ["Female", "Male"]:
    print(gender)
    print(create_tabular_data_gender(t_and_a_patient_data_peri_op, gender))

In [ ]:
for gender in ["Female", "Male"]:
    print(gender)
    perform_posthoc(t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op["GENDER"] == gender])

In [33]:
t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op["GENDER"] == "Female"].groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].mean()

**EXAMINING RACE**

In [34]:
create_histogram(t_and_a_patient_data_peri_op, "RACE_cleaned")

In [ ]:
races = t_and_a_patient_data_peri_op['RACE_cleaned'].unique()
for race in races:
    print(race)
    print(create_tabular_data_race(t_and_a_patient_data_peri_op, race))

In [ ]:
perform_posthoc(t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op['RACE_cleaned'] == "White"])
t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op['RACE_cleaned'] == "White"].groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].median()

**EXAMINING AGE GROUP**

In [ ]:
create_histogram(t_and_a_patient_data_peri_op, "age_group")

In [ ]:
age_groups = t_and_a_patient_data_peri_op['age_group'].unique()
for age_group in age_groups:
    print(age_group)
    print(create_tabular_data_age(t_and_a_patient_data_peri_op, age_group))

In [ ]:
perform_posthoc(t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op['age_group'] == "6 yr. - 11 yr."])

In [ ]:
t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op['age_group'] == "12 yr. - 18 yr."].groupby("med_combos")["HIGHEST_PACU_PAIN_SCORE"].mean()

In [ ]:
t_and_a_patient_data_peri_op[t_and_a_patient_data_peri_op['age_group'] == "12 yr. - 18 yr."]t_and_a_patient_data_peri_op["med_combos"]=="MorDex"]["HIGHEST_PACU_PAIN_SCORE"]